# Hard Endpoint Contrastive Eval / Inference

Evaluate hard-endpoint checkpoints with the unchanged pairwise + first + last permutation decoder.


In [ ]:
# 1) Install dependencies, then restart runtime once.
# After restart, run this cell again and continue.
import os
import subprocess
import sys
from pathlib import Path

MARKER = Path("/content/.snu_lgt_order_refine_deps_installed")

if not MARKER.exists():
    packages = [
        "transformers>=4.49.0,<4.54.0",
        "accelerate>=0.34.0",
        "bitsandbytes>=0.46.1",
        "peft",
        "qwen-vl-utils",
        "modelscope",
        "jedi",
        "pandas==2.2.2",
        "safetensors>=0.4.5",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *packages])
    MARKER.write_text("ok")
    print("Dependencies installed. Restarting runtime. Run this cell again after restart.")
    os.kill(os.getpid(), 9)
else:
    print("Dependencies already installed. Continue.")

In [ ]:
# 2) Setup + data unzip
from google.colab import drive
drive.mount("/content/drive")

import ast
import copy
import gc
import glob
import itertools
import json
import math
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
import random
import re
import shutil
import zipfile
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from torch.utils.data import Dataset
from transformers import AutoModelForVision2Seq, AutoProcessor, BitsAndBytesConfig, Trainer, TrainingArguments, set_seed
try:
    from transformers import Qwen2VLForConditionalGeneration
except ImportError:
    Qwen2VLForConditionalGeneration = AutoModelForVision2Seq
from peft import PeftModel, prepare_model_for_kbit_training
from transformers.utils import logging as transformers_logging

warnings.filterwarnings("ignore", category=FutureWarning, module="bitsandbytes")
warnings.filterwarnings("ignore", message=".*The following generation flags are not valid.*")
transformers_logging.set_verbosity_error()

ZIP_PATH = "/content/drive/MyDrive/SNU_AI_Challenge/snuaichallenge.zip"
DATA_DIR = "/content/snuaichallenge_data"
TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
TEST_CSV = os.path.join(DATA_DIR, "test.csv")
TRAIN_IMAGE_DIR = os.path.join(DATA_DIR, "train")
TEST_IMAGE_DIR = os.path.join(DATA_DIR, "test")

MODEL_REPO_ID = "Qwen/Qwen2-VL-2B-Instruct"
USE_MODELSCOPE_BASE_MODEL = True
DRIVE_MODEL_DIR = "/content/drive/MyDrive/SNU_AI_Challenge/model_cache/Qwen2-VL-2B-Instruct"


def ensure_base_model_path():
    if os.path.exists(os.path.join(DRIVE_MODEL_DIR, "config.json")):
        print("Using cached base model:", DRIVE_MODEL_DIR)
        return DRIVE_MODEL_DIR

    if not USE_MODELSCOPE_BASE_MODEL:
        print("Using Hugging Face repo id:", MODEL_REPO_ID)
        return MODEL_REPO_ID

    print("Base model cache not found. Downloading via ModelScope:")
    print(DRIVE_MODEL_DIR)
    from modelscope import snapshot_download as modelscope_snapshot_download

    model_dir = modelscope_snapshot_download(
        MODEL_REPO_ID,
        cache_dir="/content/modelscope_cache",
    )
    os.makedirs(os.path.dirname(DRIVE_MODEL_DIR), exist_ok=True)
    if not os.path.exists(DRIVE_MODEL_DIR):
        shutil.copytree(model_dir, DRIVE_MODEL_DIR)
    print("Base model cached at:", DRIVE_MODEL_DIR)
    return DRIVE_MODEL_DIR


def resolve_adapter_dir(base_dir):
    candidates = [
        os.path.join(base_dir, "best_adapter"),
        os.path.join(base_dir, "checkpoint-900"),
        base_dir,
    ]
    for path in candidates:
        if os.path.exists(os.path.join(path, "adapter_config.json")):
            return path
    checkpoint_dirs = sorted(
        glob.glob(os.path.join(base_dir, "checkpoint-*")),
        key=lambda path: int(re.findall(r"checkpoint-(\d+)", path)[-1]) if re.findall(r"checkpoint-(\d+)", path) else -1,
    )
    for path in reversed(checkpoint_dirs):
        if os.path.exists(os.path.join(path, "adapter_config.json")):
            return path
    raise FileNotFoundError(f"No adapter_config.json found under {base_dir}")


MODEL_ID = ensure_base_model_path()
MODEL_LOCAL_FILES_ONLY = os.path.isdir(MODEL_ID)
BASELINE_ADAPTER_DIR = (
    "/content/drive/MyDrive/SNU_AI_Challenge/"
    "qwen2vl_lgt_multitask_v1/runs/"
    "20260712_234828/lgt_multitask/checkpoint-3500"
)
REFERENCE_ADAPTER_ROOT = (
    "/content/drive/MyDrive/SNU_AI_Challenge/"
    "qwen2vl_lgt_order_refine_v1/runs/"
    "20260714_003635/lgt_order_refine"
)
REFERENCE_ADAPTER_DIR = resolve_adapter_dir(REFERENCE_ADAPTER_ROOT)
OUTPUT_ROOT = "/content/drive/MyDrive/SNU_AI_Challenge/qwen2vl_hard_endpoint_contrastive_v1"


# Set this to a specific run id when needed, for example "20260713_160355".
# If None, the latest run under OUTPUT_ROOT is used.
REFINE_RUN_ID = None


def resolve_run_root(output_root, run_id=None):
    runs_root = os.path.join(output_root, "runs")
    if run_id is not None:
        run_root = os.path.join(runs_root, run_id)
        assert os.path.isdir(run_root), run_root
        return run_root
    candidates = sorted(
        [
            os.path.join(runs_root, name)
            for name in os.listdir(runs_root)
            if os.path.isdir(os.path.join(runs_root, name))
        ]
    )
    if not candidates:
        raise RuntimeError(f"No runs found under {runs_root}")
    return candidates[-1]


RUN_ROOT = resolve_run_root(OUTPUT_ROOT, REFINE_RUN_ID)
RUN_ID = os.path.basename(RUN_ROOT)
OUTPUT_DIR = os.path.join(RUN_ROOT, "hard_endpoint_contrastive")
EVAL_DIR = os.path.join(OUTPUT_DIR, "eval")
BEST_ADAPTER_DIR = os.path.join(OUTPUT_DIR, "best_adapter")
SUBMIT_PATH = os.path.join(OUTPUT_DIR, "submission_hard_endpoint_contrastive.csv")

SEED = 42
VALID_RATIO = 0.10
TRAIN_ROWS = None
VALID_ROWS = None

RUN_CONFIG_PATH = os.path.join(RUN_ROOT, "run_config.json")
if os.path.exists(RUN_CONFIG_PATH):
    with open(RUN_CONFIG_PATH, "r", encoding="utf-8") as f:
        RUN_CONFIG = json.load(f)
else:
    RUN_CONFIG = {}
TRAIN_TASK_RATIOS = RUN_CONFIG.get("task_ratios")

QUICK_EVAL_ROWS = 50
FULL_EVAL_ROWS = 300
TOP_K_FULL_EVAL = 3
PROMPT_VERSION = "hard_endpoint_boundary_v1"

MIN_PIXELS = 128 * 28 * 28
MAX_PIXELS = 256 * 28 * 28
PAIR_INDICES = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
PERMUTATIONS = list(itertools.permutations([1, 2, 3, 4]))
ALPHAS = [0.5, 1.0, 1.5, 2.0]
BETAS = [0.0, 0.5, 1.0, 1.5, 2.0]
GAMMAS = [0.0, 0.5, 1.0, 1.5, 2.0]

for path in [EVAL_DIR, BEST_ADAPTER_DIR]:
    os.makedirs(path, exist_ok=True)

if not os.path.isdir(DATA_DIR):
    with zipfile.ZipFile(ZIP_PATH) as zip_file:
        zip_file.extractall("/content/")

assert os.path.exists(TRAIN_CSV), TRAIN_CSV
assert os.path.exists(TEST_CSV), TEST_CSV
assert os.path.isdir(TRAIN_IMAGE_DIR), TRAIN_IMAGE_DIR
assert os.path.isdir(TEST_IMAGE_DIR), TEST_IMAGE_DIR
assert os.path.exists(os.path.join(BASELINE_ADAPTER_DIR, "adapter_config.json")), BASELINE_ADAPTER_DIR
assert os.path.exists(os.path.join(REFERENCE_ADAPTER_DIR, "adapter_config.json")), REFERENCE_ADAPTER_DIR

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("run root:", RUN_ROOT)
print("baseline adapter:", BASELINE_ADAPTER_DIR)
print("reference adapter root:", REFERENCE_ADAPTER_ROOT)
print("resolved reference adapter:", REFERENCE_ADAPTER_DIR)
print("output:", OUTPUT_DIR)
print("eval:", EVAL_DIR)

In [ ]:
# 3) Data split and shared helpers
def parse_answer(answer):
    result = answer if isinstance(answer, list) else ast.literal_eval(str(answer))
    result = [int(value) for value in result]
    if len(result) != 4 or sorted(result) != [1, 2, 3, 4]:
        raise ValueError(f"Invalid Answer: {answer}")
    return result


def order_to_sequence(answer):
    return [input_index + 1 for input_index, _ in sorted(enumerate(answer), key=lambda item: item[1])]


def format_order(order):
    return "[" + ", ".join(str(int(value)) for value in order) + "]"


def parse_order_prediction(text):
    match = re.fullmatch(r"\s*\[\s*([1-4])\s*,\s*([1-4])\s*,\s*([1-4])\s*,\s*([1-4])\s*\]\s*", str(text))
    if not match:
        return None
    values = [int(value) for value in match.groups()]
    return values if sorted(values) == [1, 2, 3, 4] else None


def row_image_paths(row, image_root):
    sample_id = str(row["Id"])
    return [os.path.join(image_root, sample_id, str(row[f"Input_{i}"])) for i in range(1, 5)]


def load_rgb(path):
    with Image.open(path) as image:
        return image.convert("RGB").copy()


train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
train_df["Id"] = train_df["Id"].astype(str)
test_df["Id"] = test_df["Id"].astype(str)
train_df["Answer_list"] = train_df["Answer"].apply(parse_answer)

unique_ids = train_df["Id"].unique().copy()
rng = np.random.default_rng(SEED)
rng.shuffle(unique_ids)
valid_size = max(1, int(len(unique_ids) * VALID_RATIO))
valid_ids = set(unique_ids[:valid_size])
training_ids = set(unique_ids[valid_size:])

training_df = train_df[train_df["Id"].isin(training_ids)].reset_index(drop=True)
validation_df = train_df[train_df["Id"].isin(valid_ids)].reset_index(drop=True)

if VALID_ROWS is not None:
    validation_df = validation_df.sample(n=min(VALID_ROWS, len(validation_df)), random_state=SEED).reset_index(drop=True)

print("train/valid/test:", len(training_df), len(validation_df), len(test_df))

In [ ]:
# 4) Prompt builders, dataset, collator
def task_instruction(example):
    sentence = example["sentence"]
    task_type = example["task_type"]
    if task_type == "pairwise":
        return (
            f"Caption:\n{sentence}\n\n"
            "Question: Which image occurs first?\n"
            "If the first image occurs earlier, answer 1.\n"
            "If the second image occurs earlier, answer 2.\n"
            "Answer only 1 or 2."
        )
    if task_type == "first":
        return (
            f"Caption:\n{sentence}\n\n"
            "Question: Which image represents the beginning of the story?\n"
            "Answer only the image number from 1 to 4."
        )
    if task_type == "last":
        return (
            f"Caption:\n{sentence}\n\n"
            "Question: Which image represents the end of the story?\n"
            "Answer only the image number from 1 to 4."
        )
    if task_type == "order":
        return (
            f"Caption:\n{sentence}\n\n"
            "Question: Compare the temporal relation between scenes and identify the likely first and last scenes.\n"
            "Using these cues, determine the complete chronological order.\n"
            "Output only the final ordered list, such as [1, 2, 3, 4]."
        )
    if task_type == "first_vs_second":
        return (
            f"Full story context:\n{sentence}\n\n"
            "The two candidate images are presented in a shuffled order.\n\n"
            "Which image represents the beginning boundary of the complete chronological story?\n\n"
            "Choose the image that corresponds to the earliest scene in the full story timeline.\n"
            "The other candidate is the scene that occurs immediately after the beginning.\n\n"
            "Do not answer based on the candidate input order or the order in which events are mentioned in the caption.\n\n"
            "Use both the visual content and the full story context.\n\n"
            "Return exactly one digit:\n"
            "1 if Image 1 is the beginning scene,\n"
            "2 if Image 2 is the beginning scene."
        )
    if task_type == "third_vs_last":
        return (
            f"Full story context:\n{sentence}\n\n"
            "The two candidate images are presented in a shuffled order.\n\n"
            "Which image represents the ending boundary of the complete chronological story?\n\n"
            "Choose the image that corresponds to the final scene in the full story timeline.\n"
            "The other candidate is the scene that occurs immediately before the ending.\n\n"
            "Do not answer based on the candidate input order or the order in which events are mentioned in the caption.\n\n"
            "Use both the visual content and the full story context.\n\n"
            "Return exactly one digit:\n"
            "1 if Image 1 is the ending scene,\n"
            "2 if Image 2 is the ending scene."
        )
    raise ValueError(task_type)


def make_messages(example, include_answer=False):
    content = []
    for idx, _ in enumerate(example["image_paths"], start=1):
        content.append({"type": "text", "text": f"\nImage {idx}:"})
        content.append({"type": "image"})
    content.append({"type": "text", "text": "\n\n" + task_instruction(example)})
    messages = [{"role": "user", "content": content}]
    if include_answer:
        messages.append({"role": "assistant", "content": str(example["target"])})
    return messages


class HardEndpointContrastiveDataset(Dataset):
    def __init__(self, records):
        self.records = list(records)

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        return self.records[index]


class HardEndpointContrastiveCollator:
    def __init__(self, processor):
        self.processor = processor
        self.tokenizer = processor.tokenizer
        self.assistant_prefix_ids = self.tokenizer.encode("<|im_start|>assistant\n", add_special_tokens=False)

    def _mask_prompt(self, input_ids):
        ids = input_ids.tolist()
        labels = input_ids.clone()
        start = None
        prefix = self.assistant_prefix_ids
        for i in range(0, max(0, len(ids) - len(prefix) + 1)):
            if ids[i:i + len(prefix)] == prefix:
                start = i + len(prefix)
        if start is None:
            labels[:] = -100
        else:
            labels[:start] = -100
        labels[labels == self.tokenizer.pad_token_id] = -100
        return labels

    def __call__(self, batch):
        texts = []
        images = []
        task_types = []
        for example in batch:
            text = self.processor.apply_chat_template(make_messages(example, include_answer=True), tokenize=False, add_generation_prompt=False)
            texts.append(text)
            images.append([load_rgb(path) for path in example["image_paths"]])
            task_types.append(example["task_type"])
        encoded = self.processor(text=texts, images=images, padding=True, return_tensors="pt")
        labels = torch.stack([self._mask_prompt(row) for row in encoded["input_ids"]])
        valid_target_counts = labels.ne(-100).sum(dim=1)
        if (valid_target_counts == 0).any():
            raise ValueError(f"No assistant target tokens found: {valid_target_counts.tolist()}")
        encoded["labels"] = labels
        encoded["task_type"] = task_types
        return encoded


In [ ]:
# 5) Load processor/quant config + evaluation helpers
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
    local_files_only=MODEL_LOCAL_FILES_ONLY,
)
processor.tokenizer.padding_side = "right"
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)


def digit_token_id(digit):
    ids = processor.tokenizer.encode(str(digit), add_special_tokens=False)
    if len(ids) != 1:
        raise ValueError(f"Digit {digit} tokenized to {ids}")
    return ids[0]


DIGIT_TOKEN_IDS = {digit: digit_token_id(digit) for digit in [1, 2, 3, 4]}
EVAL_MODEL = None
LOADED_ADAPTERS = {}


def model_device(active_model):
    return next(active_model.parameters()).device


def checkpoint_name(path):
    name = os.path.basename(os.path.normpath(path))
    if path == BASELINE_ADAPTER_DIR:
        return "lgt_checkpoint_3500"
    if path == REFERENCE_ADAPTER_DIR:
        return "weighted_4task_best"
    return name


def adapter_name_for(adapter_dir):
    name = checkpoint_name(adapter_dir)
    return re.sub(r"[^0-9a-zA-Z_]+", "_", name)


def sanitize_generation_config(active_model):
    generation_config = active_model.generation_config
    generation_config.do_sample = False
    generation_config.temperature = None
    generation_config.top_p = None
    generation_config.top_k = None
    generation_config.num_beams = 1
    return active_model


def find_checkpoint_dirs(include_initial=True):
    dirs = []
    if include_initial:
        dirs.append(BASELINE_ADAPTER_DIR)
        dirs.append(REFERENCE_ADAPTER_DIR)
    if os.path.isdir(OUTPUT_DIR):
        for name in os.listdir(OUTPUT_DIR):
            path = os.path.join(OUTPUT_DIR, name)
            if name.startswith("checkpoint-") and os.path.exists(os.path.join(path, "adapter_config.json")):
                dirs.append(path)
        final_dir = os.path.join(OUTPUT_DIR, "final_adapter")
        if os.path.exists(os.path.join(final_dir, "adapter_config.json")):
            dirs.append(final_dir)

    def sort_key(path):
        if path == BASELINE_ADAPTER_DIR:
            return -2
        if path == REFERENCE_ADAPTER_DIR:
            return -1
        match = re.findall(r"checkpoint-(\d+)", path)
        if match:
            return int(match[-1])
        return 10**9

    return sorted(dict.fromkeys(dirs), key=sort_key)


def load_eval_model(adapter_dir):
    global EVAL_MODEL
    adapter_name = adapter_name_for(adapter_dir)
    if EVAL_MODEL is None:
        base = Qwen2VLForConditionalGeneration.from_pretrained(
            MODEL_ID,
            quantization_config=bnb_config,
            torch_dtype=torch.float16,
            device_map="auto",
            local_files_only=MODEL_LOCAL_FILES_ONLY,
        )
        EVAL_MODEL = PeftModel.from_pretrained(
            base,
            adapter_dir,
            adapter_name=adapter_name,
            is_trainable=False,
        )
        LOADED_ADAPTERS[adapter_dir] = adapter_name
    else:
        if adapter_dir not in LOADED_ADAPTERS:
            EVAL_MODEL.load_adapter(adapter_dir, adapter_name=adapter_name, is_trainable=False)
            LOADED_ADAPTERS[adapter_dir] = adapter_name
        EVAL_MODEL.set_adapter(LOADED_ADAPTERS[adapter_dir])
    sanitize_generation_config(EVAL_MODEL)
    EVAL_MODEL.eval()
    return EVAL_MODEL


def make_eval_example(row, task_type, pair=None):
    answer = [int(value) for value in row.get("Answer_list", [1, 2, 3, 4])]
    image_paths = row_image_paths(row, TRAIN_IMAGE_DIR)
    example = {
        "sample_id": str(row["Id"]),
        "sentence": "" if pd.isna(row["Sentence"]) else str(row["Sentence"]),
        "answer": answer,
        "order": order_to_sequence(answer),
        "image_paths": image_paths,
        "task_type": task_type,
        "target": "1",
    }
    if task_type == "pairwise":
        a, b = pair
        example["first_index"] = a - 1
        example["second_index"] = b - 1
        example["image_paths"] = [image_paths[a - 1], image_paths[b - 1]]
    return example



def make_hard_endpoint_example(row, task_type, eval_seed=SEED):
    answer = [int(value) for value in row["Answer_list"]]
    order = order_to_sequence(answer)
    image_paths = row_image_paths(row, TRAIN_IMAGE_DIR)
    rng_seed = eval_seed + int(row.name if getattr(row, "name", None) is not None else 0)
    rng = np.random.default_rng(rng_seed)
    if task_type == "first_vs_second":
        endpoint = order[0]
        near_endpoint = order[1]
    elif task_type == "third_vs_last":
        endpoint = order[-1]
        near_endpoint = order[-2]
    else:
        raise ValueError(task_type)
    if rng.random() < 0.5:
        candidate_numbers = [endpoint, near_endpoint]
        target = "1"
    else:
        candidate_numbers = [near_endpoint, endpoint]
        target = "2"
    return {
        "sample_id": str(row["Id"]),
        "sentence": "" if pd.isna(row["Sentence"]) else str(row["Sentence"]),
        "answer": answer,
        "order": order,
        "image_paths": [image_paths[int(image_number) - 1] for image_number in candidate_numbers],
        "task_type": task_type,
        "target": target,
        "candidate_image_numbers": [int(value) for value in candidate_numbers],
        "endpoint_image": int(endpoint),
        "near_endpoint_image": int(near_endpoint),
    }


@torch.no_grad()
def score_digit_candidates(active_model, example, candidates):
    old_padding_side = processor.tokenizer.padding_side
    processor.tokenizer.padding_side = "right"
    text = processor.apply_chat_template(make_messages(example, include_answer=False), tokenize=False, add_generation_prompt=True)
    images = [load_rgb(path) for path in example["image_paths"]]
    inputs = processor(text=[text], images=[images], return_tensors="pt")
    inputs = {key: value.to(model_device(active_model)) if torch.is_tensor(value) else value for key, value in inputs.items()}
    outputs = active_model(**inputs)
    last_pos = int(inputs["attention_mask"][0].sum().item()) - 1
    logits = outputs.logits[0, last_pos]
    token_ids = [DIGIT_TOKEN_IDS[int(candidate)] for candidate in candidates]
    probs = torch.softmax(logits[token_ids].float(), dim=-1).detach().cpu().numpy()
    processor.tokenizer.padding_side = old_padding_side
    return {int(candidate): float(prob) for candidate, prob in zip(candidates, probs)}


def order_ranks(order):
    return {int(image_number): position for position, image_number in enumerate(order)}


def pair_accuracy_from_orders(pred_order, gold_order):
    pred_ranks = order_ranks(pred_order)
    gold_ranks = order_ranks(gold_order)
    return np.mean([
        (pred_ranks[a] < pred_ranks[b]) == (gold_ranks[a] < gold_ranks[b])
        for a, b in itertools.combinations([1, 2, 3, 4], 2)
    ])


def order_metric_row(pred_order, gold_order):
    return {
        "exact_match": float(pred_order == gold_order),
        "pair_accuracy": float(pair_accuracy_from_orders(pred_order, gold_order)),
        "position_accuracy": float(np.mean([p == g for p, g in zip(pred_order, gold_order)])),
        "valid_output": 1.0,
    }

In [ ]:
# 6) Probability cache and structured decoding
def extract_probability_cache(adapter_dir, rows, tag):
    ckpt = checkpoint_name(adapter_dir)
    cache_path = os.path.join(EVAL_DIR, f"{ckpt}_{tag}_{PROMPT_VERSION}_task_probability_cache.json")
    if os.path.exists(cache_path):
        print("[SKIP]", cache_path)
        with open(cache_path, "r", encoding="utf-8") as f:
            return json.load(f)

    eval_model = load_eval_model(adapter_dir)
    records = []
    for _, row in tqdm(rows.iterrows(), total=len(rows), desc=f"{ckpt} {tag} probs"):
        answer = [int(value) for value in row["Answer_list"]]
        gold_order = order_to_sequence(answer)
        first_probs = score_digit_candidates(eval_model, make_eval_example(row, "first"), [1, 2, 3, 4])
        last_probs = score_digit_candidates(eval_model, make_eval_example(row, "last"), [1, 2, 3, 4])
        pair_probs = {}
        pair_correct = []
        for first_index, second_index in PAIR_INDICES:
            a, b = first_index + 1, second_index + 1
            probs = score_digit_candidates(eval_model, make_eval_example(row, "pairwise", pair=(a, b)), [1, 2])
            p_a_before_b = probs[1]
            pair_probs[f"{a}>{b}"] = float(p_a_before_b)
            pair_probs[f"{b}>{a}"] = float(1.0 - p_a_before_b)
            pred_first = a if p_a_before_b >= 0.5 else b
            gold_first = a if answer[first_index] < answer[second_index] else b
            pair_correct.append(int(pred_first == gold_first))

        first_vs_second_example = make_hard_endpoint_example(row, "first_vs_second")
        third_vs_last_example = make_hard_endpoint_example(row, "third_vs_last")
        first_vs_second_probs = score_digit_candidates(eval_model, first_vs_second_example, [1, 2])
        third_vs_last_probs = score_digit_candidates(eval_model, third_vs_last_example, [1, 2])

        records.append({
            "sample_id": str(row["Id"]),
            "gold_order": gold_order,
            "first_probs": {str(k): v for k, v in first_probs.items()},
            "last_probs": {str(k): v for k, v in last_probs.items()},
            "pair_probs": pair_probs,
            "pairwise_accuracy": float(np.mean(pair_correct)),
            "task_first_accuracy": float(max(first_probs, key=first_probs.get) == gold_order[0]),
            "task_last_accuracy": float(max(last_probs, key=last_probs.get) == gold_order[-1]),
            "first_vs_second_probs": {str(k): v for k, v in first_vs_second_probs.items()},
            "third_vs_last_probs": {str(k): v for k, v in third_vs_last_probs.items()},
            "first_vs_second_target": first_vs_second_example["target"],
            "third_vs_last_target": third_vs_last_example["target"],
            "first_vs_second_accuracy": float(max(first_vs_second_probs, key=first_vs_second_probs.get) == int(first_vs_second_example["target"])),
            "third_vs_last_accuracy": float(max(third_vs_last_probs, key=third_vs_last_probs.get) == int(third_vs_last_example["target"])),
        })

    with open(cache_path, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    return records


def structured_score(sample, order, alpha, beta, gamma):
    eps = 1e-12
    pair_score = np.mean([
        math.log(float(sample["pair_probs"][f"{order[i]}>{order[j]}"]) + eps)
        for i in range(4)
        for j in range(i + 1, 4)
    ])
    first_score = math.log(float(sample["first_probs"][str(order[0])]) + eps)
    last_score = math.log(float(sample["last_probs"][str(order[-1])]) + eps)
    return alpha * pair_score + beta * first_score + gamma * last_score


def decode_structured(sample, alpha, beta, gamma):
    return list(max(PERMUTATIONS, key=lambda order: structured_score(sample, order, alpha, beta, gamma)))


def pairwise_endpoint_scores(sample):
    eps = 1e-12
    first_scores = {}
    last_scores = {}
    for image in [1, 2, 3, 4]:
        first_scores[image] = sum(
            math.log(float(sample["pair_probs"][f"{image}>{other}"]) + eps)
            for other in [1, 2, 3, 4]
            if other != image
        )
        last_scores[image] = sum(
            math.log(float(sample["pair_probs"][f"{other}>{image}"]) + eps)
            for other in [1, 2, 3, 4]
            if other != image
        )
    return first_scores, last_scores


def evaluate_decoding(samples, alpha=1.0, beta=1.0, gamma=1.0):
    rows = []
    for sample in samples:
        gold = [int(value) for value in sample["gold_order"]]
        pred = decode_structured(sample, alpha, beta, gamma)
        metric = order_metric_row(pred, gold)
        pair_first_scores, pair_last_scores = pairwise_endpoint_scores(sample)
        direct_first = int(max(sample["first_probs"], key=lambda key: sample["first_probs"][key]))
        direct_last = int(max(sample["last_probs"], key=lambda key: sample["last_probs"][key]))
        pairwise_first = int(max(pair_first_scores, key=pair_first_scores.get))
        pairwise_last = int(max(pair_last_scores, key=pair_last_scores.get))
        metric.update({
            "sample_id": sample["sample_id"],
            "pred_order": pred,
            "gold_order": gold,
            "pairwise_accuracy": sample["pairwise_accuracy"],
            "task_first_accuracy": sample["task_first_accuracy"],
            "task_last_accuracy": sample["task_last_accuracy"],
            "task_first_last_both_correct": float(sample["task_first_accuracy"] == 1.0 and sample["task_last_accuracy"] == 1.0),
            "decoded_first_accuracy": float(pred[0] == gold[0]),
            "decoded_last_accuracy": float(pred[-1] == gold[-1]),
            "decoded_first_last_both_correct": float(pred[0] == gold[0] and pred[-1] == gold[-1]),
            "direct_first": direct_first,
            "pairwise_first": pairwise_first,
            "direct_last": direct_last,
            "pairwise_last": pairwise_last,
            "first_endpoint_agree": float(direct_first == pairwise_first),
            "last_endpoint_agree": float(direct_last == pairwise_last),
            "first_vs_second_accuracy": sample.get("first_vs_second_accuracy", np.nan),
            "third_vs_last_accuracy": sample.get("third_vs_last_accuracy", np.nan),
        })
        rows.append(metric)
    df = pd.DataFrame(rows)
    summary = {
        "exact_match": df["exact_match"].mean(),
        "pair_accuracy": df["pair_accuracy"].mean(),
        "position_accuracy": df["position_accuracy"].mean(),
        "valid_output_rate": df["valid_output"].mean(),
        "mean_pairwise_accuracy": df["pairwise_accuracy"].mean(),
        "task_first_accuracy": df["task_first_accuracy"].mean(),
        "task_last_accuracy": df["task_last_accuracy"].mean(),
        "task_first_last_both_correct_rate": df["task_first_last_both_correct"].mean(),
        "decoded_first_accuracy": df["decoded_first_accuracy"].mean(),
        "decoded_last_accuracy": df["decoded_last_accuracy"].mean(),
        "decoded_first_last_both_correct_rate": df["decoded_first_last_both_correct"].mean(),
        "exact_given_task_first_last_correct": df.loc[df["task_first_last_both_correct"] == 1.0, "exact_match"].mean() if (df["task_first_last_both_correct"] == 1.0).any() else np.nan,
        "exact_given_decoded_first_last_correct": df.loc[df["decoded_first_last_both_correct"] == 1.0, "exact_match"].mean() if (df["decoded_first_last_both_correct"] == 1.0).any() else np.nan,
        "first_endpoint_agreement": df["first_endpoint_agree"].mean(),
        "last_endpoint_agreement": df["last_endpoint_agree"].mean(),
        "first_vs_second_accuracy": df["first_vs_second_accuracy"].mean() if "first_vs_second_accuracy" in df else np.nan,
        "third_vs_last_accuracy": df["third_vs_last_accuracy"].mean() if "third_vs_last_accuracy" in df else np.nan,
    }
    return summary, df


def grid_search(samples, checkpoint, tag):
    path = os.path.join(EVAL_DIR, f"{checkpoint}_{tag}_{PROMPT_VERSION}_decoding_weight_search.csv")
    if os.path.exists(path):
        return pd.read_csv(path)
    rows = []
    for alpha, beta, gamma in itertools.product(ALPHAS, BETAS, GAMMAS):
        summary, _ = evaluate_decoding(samples, alpha=alpha, beta=beta, gamma=gamma)
        summary.update({"checkpoint": checkpoint, "tag": tag, "alpha": alpha, "beta": beta, "gamma": gamma, "decoding": "pair_first_last"})
        rows.append(summary)
    df = pd.DataFrame(rows).sort_values(["exact_match", "pair_accuracy", "position_accuracy"], ascending=False).reset_index(drop=True)
    df.to_csv(path, index=False)
    return df

In [ ]:
# 7) Quick checkpoint evaluation: pairwise + first + last structured decoding only
quick_rows = validation_df.sample(n=min(QUICK_EVAL_ROWS, len(validation_df)), random_state=SEED).reset_index(drop=True)
checkpoint_dirs = find_checkpoint_dirs(include_initial=True)
print("checkpoints:", [checkpoint_name(path) for path in checkpoint_dirs])

summary_path = os.path.join(EVAL_DIR, f"all_checkpoint_metrics_quick_{PROMPT_VERSION}.csv")
if os.path.exists(summary_path):
    summary_df = pd.read_csv(summary_path)
    completed = set(summary_df["checkpoint"].astype(str))
    summary_rows = summary_df.to_dict("records")
else:
    completed = set()
    summary_rows = []

for adapter_dir in checkpoint_dirs:
    ckpt = checkpoint_name(adapter_dir)
    if ckpt in completed:
        print("[SKIP]", ckpt)
        continue
    samples = extract_probability_cache(adapter_dir, quick_rows, tag=f"quick{len(quick_rows)}")
    search = grid_search(samples, ckpt, tag=f"quick{len(quick_rows)}")
    best_structured = search.iloc[0].to_dict()
    summary_rows.append(best_structured)
    pd.DataFrame(summary_rows).to_csv(summary_path, index=False)

summary_df = pd.DataFrame(summary_rows).sort_values(["exact_match", "pair_accuracy", "position_accuracy"], ascending=False).reset_index(drop=True)
display(summary_df)
top_checkpoints = summary_df.head(TOP_K_FULL_EVAL)["checkpoint"].tolist()
print("top structured checkpoints:", top_checkpoints)

In [ ]:
# 8) Full validation with tuning/holdout split
shuffled_validation = validation_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
tuning_rows = shuffled_validation.iloc[:min(150, len(shuffled_validation))].reset_index(drop=True)
holdout_rows = shuffled_validation.iloc[min(150, len(shuffled_validation)):min(300, len(shuffled_validation))].reset_index(drop=True)
if len(holdout_rows) == 0:
    raise RuntimeError("Holdout rows are empty. Need at least 151 validation rows.")

full_summary_path = os.path.join(EVAL_DIR, f"all_checkpoint_metrics_full300_{PROMPT_VERSION}.csv")
if os.path.exists(full_summary_path):
    full_summary_df = pd.read_csv(full_summary_path)
    completed = set(full_summary_df["checkpoint"].astype(str))
    full_rows_out = full_summary_df.to_dict("records")
else:
    completed = set()
    full_rows_out = []

name_to_dir = {checkpoint_name(path): path for path in checkpoint_dirs}
baseline_names = [checkpoint_name(BASELINE_ADAPTER_DIR), checkpoint_name(REFERENCE_ADAPTER_DIR)]
new_checkpoint_names = [
    name
    for name in summary_df["checkpoint"].tolist()
    if name not in set(baseline_names)
]
top_new_checkpoints = new_checkpoint_names[:TOP_K_FULL_EVAL]
full_checkpoint_names = list(dict.fromkeys(baseline_names + top_new_checkpoints))
print("full checkpoints:", full_checkpoint_names)

for ckpt in full_checkpoint_names:
    adapter_dir = name_to_dir[ckpt]
    if ckpt in completed:
        print("[SKIP]", ckpt)
        continue

    tuning_samples = extract_probability_cache(adapter_dir, tuning_rows, tag=f"tuning{len(tuning_rows)}")
    holdout_samples = extract_probability_cache(adapter_dir, holdout_rows, tag=f"holdout{len(holdout_rows)}")
    search = grid_search(tuning_samples, ckpt, tag=f"tuning{len(tuning_rows)}")
    best_weights = search.iloc[0].to_dict()
    holdout_summary, structured_predictions = evaluate_decoding(
        holdout_samples,
        alpha=best_weights["alpha"],
        beta=best_weights["beta"],
        gamma=best_weights["gamma"],
    )
    holdout_summary.update({
        "checkpoint": ckpt,
        "tag": f"holdout{len(holdout_rows)}",
        "alpha": best_weights["alpha"],
        "beta": best_weights["beta"],
        "gamma": best_weights["gamma"],
        "decoding": "pair_first_last",
        "tuning_exact_match": best_weights["exact_match"],
        "tuning_pair_accuracy": best_weights["pair_accuracy"],
        "tuning_position_accuracy": best_weights["position_accuracy"],
    })

    structured_predictions.to_csv(os.path.join(EVAL_DIR, f"{ckpt}_holdout_checkpoint_predictions_{PROMPT_VERSION}.csv"), index=False)
    structured_predictions[[
        "sample_id",
        "exact_match",
        "direct_first",
        "pairwise_first",
        "first_endpoint_agree",
        "direct_last",
        "pairwise_last",
        "last_endpoint_agree",
    ]].to_csv(os.path.join(EVAL_DIR, f"{ckpt}_endpoint_agreement_analysis_{PROMPT_VERSION}.csv"), index=False)
    full_rows_out.append(holdout_summary)
    pd.DataFrame(full_rows_out).to_csv(full_summary_path, index=False)

full_summary_df = pd.DataFrame(full_rows_out).sort_values(["exact_match", "pair_accuracy", "position_accuracy"], ascending=False).reset_index(drop=True)
display(full_summary_df)

best = full_summary_df.iloc[0].to_dict()
best_checkpoint = best["checkpoint"]
best_adapter_dir = name_to_dir[best_checkpoint]
print("BEST:", best)

if os.path.exists(BEST_ADAPTER_DIR) and not os.path.exists(os.path.join(BEST_ADAPTER_DIR, "adapter_config.json")):
    shutil.rmtree(BEST_ADAPTER_DIR)
os.makedirs(BEST_ADAPTER_DIR, exist_ok=True)
if not os.path.exists(os.path.join(BEST_ADAPTER_DIR, "adapter_config.json")):
    for filename in os.listdir(best_adapter_dir):
        src = os.path.join(best_adapter_dir, filename)
        dst = os.path.join(BEST_ADAPTER_DIR, filename)
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
        else:
            shutil.copy2(src, dst)
    processor.save_pretrained(BEST_ADAPTER_DIR)

best_config = {
    "initial_adapter": RUN_CONFIG.get("initial_adapter_dir"),
    "checkpoint": best_checkpoint,
    "checkpoint_dir": best_adapter_dir,
    "prompt_version": PROMPT_VERSION,
    "task_ratios": TRAIN_TASK_RATIOS,
    "task_loss_weights": RUN_CONFIG.get("task_loss_weights"),
    "experiment": "hard_endpoint_contrastive_v1",
    "baseline_test_score": 0.56544,
    "decoding": {
        "type": "pair_first_last_permutation_search",
        "alpha": float(best["alpha"]),
        "beta": float(best["beta"]),
        "gamma": float(best["gamma"]),
    },
    "holdout_exact_match": float(best["exact_match"]),
    "holdout_pair_accuracy": float(best["pair_accuracy"]),
    "holdout_position_accuracy": float(best["position_accuracy"]),
}
with open(os.path.join(BEST_ADAPTER_DIR, "best_config.json"), "w", encoding="utf-8") as f:
    json.dump(best_config, f, ensure_ascii=False, indent=2)
shutil.copy2(full_summary_path, os.path.join(BEST_ADAPTER_DIR, f"all_checkpoint_metrics_full300_{PROMPT_VERSION}.csv"))
print("best adapter saved:", BEST_ADAPTER_DIR)

In [ ]:
# 9) Test inference and submission
def sequence_to_answer(order):
    answer = [0] * 4
    for rank, image_number in enumerate(order, start=1):
        answer[int(image_number) - 1] = rank
    return answer


def make_test_example(row, task_type, pair=None):
    image_paths = row_image_paths(row, TEST_IMAGE_DIR)
    example = {
        "sample_id": str(row["Id"]),
        "sentence": "" if pd.isna(row["Sentence"]) else str(row["Sentence"]),
        "answer": [1, 2, 3, 4],
        "order": [1, 2, 3, 4],
        "image_paths": image_paths,
        "task_type": task_type,
        "target": "1",
    }
    if task_type == "pairwise":
        a, b = pair
        example["image_paths"] = [image_paths[a - 1], image_paths[b - 1]]
    return example


@torch.no_grad()
def test_digit_probs(active_model, row, task_type, candidates, pair=None):
    example = make_test_example(row, task_type, pair=pair)
    return score_digit_candidates(active_model, example, candidates)


with open(os.path.join(BEST_ADAPTER_DIR, "best_config.json"), "r", encoding="utf-8") as f:
    best_config = json.load(f)
decode_config = best_config["decoding"]

test_model = load_eval_model(BEST_ADAPTER_DIR)
submission_rows = []
test_cache = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="test inference"):
    first_probs = test_digit_probs(test_model, row, "first", [1, 2, 3, 4])
    last_probs = test_digit_probs(test_model, row, "last", [1, 2, 3, 4])
    pair_probs = {}
    for first_index, second_index in PAIR_INDICES:
        a, b = first_index + 1, second_index + 1
        probs = test_digit_probs(test_model, row, "pairwise", [1, 2], pair=(a, b))
        pair_probs[f"{a}>{b}"] = float(probs[1])
        pair_probs[f"{b}>{a}"] = float(1.0 - probs[1])
    sample = {
        "sample_id": str(row["Id"]),
        "first_probs": {str(k): v for k, v in first_probs.items()},
        "last_probs": {str(k): v for k, v in last_probs.items()},
        "pair_probs": pair_probs,
    }
    pred_order = decode_structured(
        sample,
        alpha=float(decode_config["alpha"]),
        beta=float(decode_config["beta"]),
        gamma=float(decode_config["gamma"]),
    )
    test_cache.append(sample | {"pred_order": pred_order})
    submission_rows.append({"Id": str(row["Id"]), "Answer": str(sequence_to_answer(pred_order))})

submission = pd.DataFrame(submission_rows)
submission_path = SUBMIT_PATH
submission.to_csv(submission_path, index=False)
with open(os.path.join(EVAL_DIR, f"test_probability_cache_{PROMPT_VERSION}.json"), "w", encoding="utf-8") as f:
    json.dump(test_cache, f, ensure_ascii=False, indent=2)
shutil.copy2(submission_path, os.path.join(BEST_ADAPTER_DIR, "submission.csv"))
display(submission.head())
print("submission saved:", submission_path)

## Outputs

```text
eval/all_checkpoint_metrics_quick_{PROMPT_VERSION}.csv
eval/all_checkpoint_metrics_full300_{PROMPT_VERSION}.csv
eval/*_{PROMPT_VERSION}_task_probability_cache.json
eval/*_decoding_weight_search.csv
eval/*_checkpoint_predictions_{PROMPT_VERSION}.csv
eval/*_endpoint_agreement_analysis_{PROMPT_VERSION}.csv
best_adapter/best_config.json
submission_hard_endpoint_contrastive.csv
```